# Magic (Dunder) Methods

You have likely noticed that Python makes heavy use of methods wrapped in Double UNDERscores (hence the term "Dunder" methods). We've already used __init__ extensively.

These methods are Python's way of hooking your custom objects into the core language syntax. When you use operators like +, ==, len(), or print(), Python is secretly calling specific dunder methods behind the scenes.

By implementing these methods in your class, you can make your custom objects behave exactly like built-in Python data types.

## 1. String Representations: __str__ vs. __repr__

- `__str__(self)`: The "informal" string. This is meant to be highly readable for the end-user. It is called by `print(obj)` and `str(obj)`.

- `__repr__(self)`: The "official" string. This is meant for developers and debugging. It should ideally be a string that, if pasted back into Python, would recreate the object. It is called when you inspect an object in the REPL console or call `repr(obj)`.

In [18]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # For the user: "Vector at X: 5, Y: 10"
    def __str__(self):
        return f"Vector at X: {self.x}, Y: {self.y}"

    # For the developer: "Vector(5, 10)"
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v = Vector(5, 10)

print(v)       # Calls __str__ -> Vector at X: 5, Y: 10
# If you were in a terminal and just typed 'v' and hit enter, 
# it would call __repr__ -> Vector(5, 10)

Vector at X: 5, Y: 10


## 2. Operator Overloading
You can redefine what math and comparison operators do when applied to your objects.

Let's say we want to be able to add two Vector objects together using the standard + symbol, and check if they are identical using ==.

In [19]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # Hooks into the '+' operator
    def __add__(self, other_vector):
        if not isinstance(other_vector, Vector):
            return NotImplemented
        
        # Return a brand new Vector instance
        new_x = self.x + other_vector.x
        new_y = self.y + other_vector.y
        return Vector(new_x, new_y)

    # Hooks into the '==' operator
    def __eq__(self, other_vector):
        if not isinstance(other_vector, Vector):
            return False
        return self.x == other_vector.x and self.y == other_vector.y
        
    def __repr__(self):
         return f"Vector({self.x}, {self.y})"

v1 = Vector(2, 4)
v2 = Vector(1, 3)

# Python sees '+', so it calls v1.__add__(v2)
v3 = v1 + v2 
print(v3) # Output: Vector(3, 7)

# Python sees '==', so it calls v3.__eq__(Vector(3, 7))
print(v3 == Vector(3, 7)) # Output: True

Vector(3, 7)
True


## 3. Context Managers: `__enter__` and `__exit__`
This is arguably the most important dunder feature for writing production-level backend code (like your FastAPI or Spring Boot services).

When you deal with resources like database connections, open files, or network sockets, you must ensure they are closed when you are done, even if an error occurs. Python provides the with statement for this.

To make your class work with with, you implement __enter__ (setup) and __exit__ (teardown).

In [20]:
class DatabaseConnection:
    def __init__(self, db_url):
        self.db_url = db_url
        self.connection = None

    def __enter__(self):
        print(f"Opening connection to {self.db_url}...")
        self.connection = "Active DB Socket"
        # What we return here is assigned to the 'as' variable
        return self 

    def __exit__(self, exc_type, exc_value, traceback):
        # This ALWAYS runs, even if an exception happens inside the block
        print(f"Closing connection safely.")
        self.connection = None
        # If an error occurred, exc_type will contain the error info

# Usage:
with DatabaseConnection("postgresql://localhost:5432") as db:
    print(f"Inside block. Status: {db.connection}")
    # Simulating work...
    # Even if an error happens here, __exit__ is guaranteed to run!

# Output:
# Opening connection to postgresql://localhost:5432...
# Inside block. Status: Active DB Socket
# Closing connection safely.

Opening connection to postgresql://localhost:5432...
Inside block. Status: Active DB Socket
Closing connection safely.


## 4. Callables: `__call__`
Finally, you can make an instance of your class behave like a function. If you implement `__call__`, you can invoke the object with parentheses ().

This is heavily used in AI/Machine Learning frameworks (like LangChain/LangGrap) where models or agents are instantiated objects that you "call" with input data.

In [21]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

# We create the object, saving state (factor = 5)
times_five = Multiplier(5)

# We CALL the object like a function!
print(times_five(10)) # Output: 50

50
